# Canadian Transaction Classifier (CTC)

**Author:** Mohit Gummaraj Kishore
**Version:** 2.1.0

| Build | Change |
|-------|--------|
| 1.0.0 | Initial build: 5-model pipeline, ONNX export, section numbering |
| 1.1.0 | Multi-country data pipeline, T2125 line number corrections, current/capital expense types |
| 1.2.0 | RunPod support: environment detection, /workspace paths, MI300X tuning |
| 1.3.0 | OOM fix: lazy per-sample tokenization, stratified sampling, gradient checkpointing |
| 1.4.0 | Fix: skip torch/onnxruntime reinstall on RunPod to preserve ROCm build |
| 1.5.0 | Fix: auto-restore ROCm torch guard, GPU sanity check before transformers import |
| 1.6.0 | Fix: pin transformers<4.46 for torch 2.0.1 compatibility on RunPod |
| 1.7.0 | Fix: remove accelerate to break boto3/botocore/urllib3 conflict chain |
| 1.8.0 | Add version table to notebook header |
| 1.9.0 | Fix: HSA_OVERRIDE_GFX_VERSION for MI300X, CPU fallback on HIP kernel errors |
| 2.0.0 | Fix: nbstripout git filter so cell outputs never cause pull conflicts |
| 2.1.0 | Fix: pin scientific stack versions; pod restarts wipe pip installs |

---

A comparative research project evaluating classical ML and transformer models for classifying Canadian bank transactions into CRA T2125 expense categories.

**Research Questions:**
1. Will transformers perform better, or will classical ML algorithms be sufficient for transaction classification?
2. Which model is best suited for CRA T2125 expense classification for independent contractors and small business owners?
3. Which model performs best for browser and on-device inference (iPhone, Mac)?

**Models Evaluated:** Logistic Regression, Linear SVM, LightGBM, DistilBERT, MobileBERT

**Dataset:** [mitulshah/transaction-categorization](https://huggingface.co/datasets/mitulshah/transaction-categorization) on HuggingFace

## 1.0 Environment Setup

This section detects whether the notebook runs in Google Colab, RunPod, or a local machine, configures project paths accordingly, installs dependencies, and imports all required libraries.

Supported environments:
- **RunPod**: Detected via the `RUNPOD_POD_ID` environment variable. Project files are written to `/workspace/ctc_project` on the persistent volume.
- **Google Colab**: Detected via the `google.colab` module. Google Drive is mounted for persistent storage.
- **Local**: Falls back to a configurable directory on the local filesystem.

### 1.1 Detect Environment and Configure Paths

Detects the execution environment in priority order: RunPod, Google Colab, then local.

- **RunPod**: Uses `/workspace/ctc_project`. The `/workspace` volume is persistent across pod restarts. No extra setup is required.
- **Colab**: Mounts Google Drive and uses a folder in your Drive.
- **Local**: Uses the `LOCAL_PROJECT_DIR` path defined at the top of this cell.

In [ ]:
import os
import sys

# ------------------------------------------------------------
# LOCAL PATH CONFIGURATION - edit this for local runs only
# ------------------------------------------------------------
LOCAL_PROJECT_DIR = os.path.expanduser('~/ctc_project')
# ------------------------------------------------------------

IS_COLAB = False
IS_RUNPOD = False

# RunPod sets RUNPOD_POD_ID in every container environment
if os.environ.get('RUNPOD_POD_ID'):
    IS_RUNPOD = True
    project_dir = '/workspace/ctc_project'
    print(f'[OK] RunPod environment detected. Pod ID: {os.environ["RUNPOD_POD_ID"]}')
    print(f'[OK] Project directory: {project_dir} (persistent /workspace volume)')
else:
    try:
        import google.colab  # noqa: F401
        IS_COLAB = True
    except ImportError:
        pass

    if IS_COLAB:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            project_dir = '/content/drive/MyDrive/ctc_project'
            print('[OK] Google Drive mounted')
        except Exception as e:
            print(f'[WARNING] Could not mount Google Drive: {e}')
            project_dir = '/content/ctc_project'
    else:
        project_dir = LOCAL_PROJECT_DIR
        print(f'[INFO] Running locally. Project dir: {project_dir}')

# Create required subdirectories
for subdir in ['results', 'models', 'charts']:
    path = os.path.join(project_dir, subdir)
    try:
        os.makedirs(path, exist_ok=True)
    except Exception as e:
        print(f'[WARNING] Could not create {path}: {e}')

print(f'[OK] Project directory: {project_dir}')
print(f'[OK] Results:  {project_dir}/results')
print(f'[OK] Models:   {project_dir}/models')
print(f'[OK] Charts:   {project_dir}/charts')


### 1.2 Install Dependencies

Installs all required Python packages. Packages that are already installed are skipped by pip automatically.

**RunPod note:** `torch` and `onnxruntime` are intentionally excluded from the install list on RunPod. The container image (`runpod/pytorch:2.0.1-py3.10-rocm5.7-ubuntu22.04`) ships a ROCm-specific PyTorch build. Running `pip install torch` overwrites it with the standard CPU/CUDA build from PyPI, which breaks the AMD GPU support and causes transformers to report 'PyTorch not found'. `onnxruntime-rocm` is the correct variant for this hardware and is pre-installed.

In [ ]:
import subprocess

# On RunPod (ROCm), verify torch is the ROCm build before installing anything.
# A previous run may have overwritten the ROCm torch with the standard PyPI build.
# Detect this and restore it before other packages are installed.
if IS_RUNPOD:
    try:
        import torch as _torch
        _is_rocm = hasattr(_torch.version, 'hip') and _torch.version.hip is not None
        if not _is_rocm:
            print('[WARNING] Standard (non-ROCm) torch detected. Restoring ROCm build...')
            subprocess.check_call([
                sys.executable, '-m', 'pip', 'install', '-q',
                'torch', '--index-url', 'https://download.pytorch.org/whl/rocm5.7'
            ])
            print('[OK] ROCm torch restored. Restart the kernel if imports fail below.')
        else:
            print('[OK] ROCm torch already in place.')
    except Exception as _e:
        print(f'[WARNING] Could not verify torch build: {_e}')

# Install nbstripout once per repo clone so cell outputs are never committed.
# This prevents git pull conflicts caused by Jupyter saving outputs into the .ipynb file.
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nbstripout'])
    subprocess.check_call(['nbstripout', '--install'], cwd=os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '/workspace/ai-applications')
    print('[OK] nbstripout installed and git filter active -- outputs will not be committed.')
except Exception as _nbs_err:
    print(f'[WARNING] nbstripout setup failed: {_nbs_err}')
    print('       Run manually: pip install nbstripout && nbstripout --install')

# Packages installed on all environments.
# The RunPod container ships numpy 1.21.2, scipy 1.8.1, and sklearn/matplotlib
# versions that are incompatible with each other after any numpy upgrade.
# Pinning the full scientific stack here ensures every pod restart gets a
# consistent, mutually compatible set of versions regardless of container defaults.
PACKAGES_COMMON = [
    # Scientific stack -- pinned together for mutual compatibility
    "numpy>=1.23,<2.0",
    "scipy>=1.10.0,<2.0",
    "scikit-learn>=1.3.0",
    "matplotlib>=3.6.0",
    "seaborn>=0.12.0",
    "pandas>=1.5.0",
    # ML and data
    "datasets>=2.14.0",
    "lightgbm>=3.3.0",
    # Transformers -- pinned <4.46 for torch 2.0.1 compatibility on RunPod
    "transformers>=4.35.0,<4.46.0" if IS_RUNPOD else "transformers>=4.35.0",
    # Export and utilities
    "onnx",
    "skl2onnx",
    "optimum[exporters]",
    "tqdm",
    "nbstripout",
    # accelerate excluded: pulls boto3->botocore->urllib3<2.0 which breaks imports
]

# torch and onnxruntime are excluded on RunPod -- the ROCm builds are pre-installed
# and must not be overwritten by the standard PyPI packages.
PACKAGES_NON_RUNPOD = [
    "torch",
    "onnxruntime",
]

packages_to_install = PACKAGES_COMMON[:]
if IS_RUNPOD:
    print('[INFO] RunPod: skipping torch and onnxruntime (pre-installed ROCm builds).')
else:
    packages_to_install += PACKAGES_NON_RUNPOD

failed = []
for pkg in packages_to_install:
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            stderr=subprocess.DEVNULL,
        )
        print(f'[OK] {pkg}')
    except subprocess.CalledProcessError:
        print(f'[FAILED] {pkg}')
        failed.append(pkg)

if failed:
    print(f'\n[WARNING] {len(failed)} package(s) failed to install: {failed}')
else:
    print('\n[OK] All packages installed successfully.')


### 1.3 Import Libraries

All imports are grouped by purpose. Import errors are collected and reported; a summary at the end indicates whether critical dependencies are available.

In [ ]:
import_errors = []

# Core
try:
    import pandas as pd
    import numpy as np
    print('[OK] pandas, numpy')
except ImportError as e:
    import_errors.append(str(e))
    print(f'[FAILED] pandas/numpy: {e}')

# Visualization
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    print('[OK] matplotlib, seaborn')
except ImportError as e:
    import_errors.append(str(e))
    print(f'[FAILED] matplotlib/seaborn: {e}')

# Scikit-learn
try:
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score,
        f1_score, classification_report, confusion_matrix
    )
    print('[OK] scikit-learn')
except ImportError as e:
    import_errors.append(str(e))
    print(f'[FAILED] scikit-learn: {e}')

# LightGBM
try:
    from lightgbm import LGBMClassifier
    print('[OK] lightgbm')
except ImportError as e:
    import_errors.append(str(e))
    print(f'[FAILED] lightgbm: {e}')

# PyTorch -- verify the ROCm build is active before importing transformers
try:
    import torch
    from torch.utils.data import DataLoader, Dataset
    _is_rocm = hasattr(torch.version, 'hip') and torch.version.hip is not None
    _gpu_ok = torch.cuda.is_available()
    print(f'[OK] torch {torch.__version__} | ROCm={_is_rocm} | CUDA available={_gpu_ok}')
    if IS_RUNPOD and not _is_rocm:
        print('[ERROR] Standard (non-ROCm) torch is active on RunPod.')
        print('        Run this in a terminal, then restart the kernel:')
        print('        pip install torch --index-url https://download.pytorch.org/whl/rocm5.7')
        import_errors.append('torch: non-ROCm build detected on RunPod')
except ImportError as e:
    import_errors.append(str(e))
    print(f'[FAILED] torch: {e}')

# Transformers -- import only after torch check
try:
    from transformers import (
        DistilBertTokenizer,
        DistilBertForSequenceClassification,
        MobileBertTokenizer,
        MobileBertForSequenceClassification,
        get_linear_schedule_with_warmup,
    )
    print('[OK] transformers')
except ImportError as e:
    import_errors.append(str(e))
    print(f'[FAILED] transformers: {e}')

# ONNX
try:
    import onnx
    import onnxruntime as ort
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType, StringTensorType
    print('[OK] onnx, onnxruntime, skl2onnx')
except ImportError as e:
    import_errors.append(str(e))
    print(f'[WARNING] onnx/skl2onnx not fully available: {e}')

# Utilities
import json
import time
import pickle
import os
import gc
from tqdm.auto import tqdm

print('[OK] Utility imports (json, time, pickle, os, gc, tqdm)')

if import_errors:
    print(f'\n[WARNING] {len(import_errors)} import error(s) detected. Review messages above.')
else:
    print('\n[OK] All imports successful.')


### 1.4 Detect Compute Device

Detects the best available compute device: CUDA/ROCm GPU, Apple MPS, or CPU.

**AMD GPU (RunPod MI300X):** PyTorch with ROCm exposes AMD GPUs via the CUDA API. `torch.cuda.is_available()` returns `True` on ROCm and `torch.cuda.get_device_name(0)` returns the AMD device name.

**HIP kernel compatibility:** The MI300X uses architecture `gfx942`. If PyTorch ROCm 5.7 was not compiled with `gfx942` kernels, every GPU operation throws `HIP error: invalid device function`. The cell sets `HSA_OVERRIDE_GFX_VERSION=9.4.2` at startup to signal the correct architecture to ROCm. If GPU errors still occur during training, `train_transformer` automatically falls back to CPU.

In [ ]:
try:
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        is_rocm = hasattr(torch.version, 'hip') and torch.version.hip is not None
        device_type = 'AMD ROCm GPU' if is_rocm else 'NVIDIA CUDA GPU'

        if is_rocm:
            # MI300X uses gfx942. ROCm 5.7 may not include gfx942 kernels by default.
            # HSA_OVERRIDE_GFX_VERSION tells the ROCm runtime which arch to target.
            os.environ.setdefault('HSA_OVERRIDE_GFX_VERSION', '9.4.2')
            print(f'[OK] HSA_OVERRIDE_GFX_VERSION = {os.environ["HSA_OVERRIDE_GFX_VERSION"]}')

            # Validate GPU kernels work with a small tensor op before committing to GPU training
            try:
                _t = torch.tensor([1.0], device='cuda') * 2
                del _t
                gpu_kernels_ok = True
                print('[OK] GPU kernel probe passed -- gfx942 kernels are functional.')
            except RuntimeError as _hip_err:
                gpu_kernels_ok = False
                print(f'[WARNING] GPU kernel probe failed: {_hip_err}')
                print('[INFO] Transformer training will run on CPU instead.')

            if gpu_kernels_ok:
                device = torch.device('cuda')
            else:
                device = torch.device('cpu')
                device_type = 'CPU (GPU kernel fallback)'
        else:
            device = torch.device('cuda')
            gpu_kernels_ok = True

    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
        device_type = 'Apple MPS'
        device_name = 'Apple Silicon GPU'
        gpu_kernels_ok = True
    else:
        device = torch.device('cpu')
        device_type = 'CPU'
        device_name = 'CPU'
        gpu_kernels_ok = False

    print(f'[OK] Device type:  {device_type}')
    print(f'[OK] Device name:  {device_name}')
    print(f'[OK] torch.device: {device}')

    if torch.cuda.is_available() and device.type == 'cuda':
        total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[OK] GPU memory:   {total_mem:.1f} GB')

except Exception as e:
    print(f'[WARNING] Device detection failed: {e}. Falling back to CPU.')
    device = torch.device('cpu')
    device_type = 'CPU'
    device_name = 'CPU'
    gpu_kernels_ok = False


## 2.0 Configuration

A single configuration cell defines all paths, hyperparameters, and feature flags for the notebook. Edit values here to customize behaviour without touching downstream cells.

### 2.1 Global Configuration Cell

Flags under `ENABLE_MODELS` let you skip individual models. Set any flag to `False` to skip that model.

**RunPod MI300X tuning:** The defaults below are set for RunPod with an MI300X (192 GB VRAM, 283 GB RAM):
- `TRANSFORMER_BATCH_SIZE = 64`: safe for MI300X; increase to 128 if VRAM headroom allows.
- `TRANSFORMER_MAX_TRAIN_SAMPLES = 500000`: uses 500k stratified samples for transformer fine-tuning. Set to `None` to train on the full 3.6M rows (requires the full 283 GB RAM).

If you run on a lower-memory machine, set `TRANSFORMER_BATCH_SIZE = 16` and `TRANSFORMER_MAX_TRAIN_SAMPLES = 75000`.

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION - edit this cell to customize the run
# ============================================================

# HuggingFace authentication
# RunPod: set via pod environment variables in the RunPod dashboard (see setup guide below).
# Colab:  set via Colab Secrets (left sidebar -> key icon).
# Local:  export HF_TOKEN=hf_... in your shell before launching Jupyter.
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        if IS_COLAB:
            from google.colab import userdata
            HF_TOKEN = userdata.get('HF_TOKEN', '')
    except Exception:
        pass
if not HF_TOKEN:
    print('[WARNING] HF_TOKEN not set. Private datasets will be inaccessible.')
else:
    print('[OK] HuggingFace token loaded.')

# Dataset
DATASET_NAME = 'mitulshah/transaction-categorization'
TEST_SIZE = 0.20
RANDOM_SEED = 42

# TF-IDF settings (shared by LR, SVM, LightGBM)
TFIDF_MAX_FEATURES = 5000
TFIDF_NGRAM_RANGE = (1, 2)

# Classical ML hyperparameters
LR_MAX_ITER = 1000
SVM_MAX_ITER = 5000
LGBM_N_ESTIMATORS = 200
LGBM_NUM_LEAVES = 63
LGBM_LR = 0.05

# Transformer hyperparameters
# Defaults are tuned for RunPod MI300X (192 GB VRAM, 283 GB RAM).
# For Colab free tier, use TRANSFORMER_BATCH_SIZE=16, TRANSFORMER_MAX_TRAIN_SAMPLES=75000.
TRANSFORMER_MAX_LENGTH = 128
TRANSFORMER_BATCH_SIZE = 64       # MI300X: safe at 64, can push to 128
TRANSFORMER_EPOCHS = 3
TRANSFORMER_LR = 5e-5
TRANSFORMER_MAX_TRAIN_SAMPLES = 500000  # Set to None to use all 3.6M rows

# Model flags - set False to skip a model
ENABLE_MODELS = {
    'logistic_regression': True,
    'linear_svm': True,
    'lightgbm': True,
    'distilbert': True,
    'mobilebert': True,
}

print('[OK] Configuration loaded.')
print(f'  Dataset:                    {DATASET_NAME}')
print(f'  Test split:                 {TEST_SIZE}')
print(f'  Random seed:                {RANDOM_SEED}')
print(f'  Transformer LR:             {TRANSFORMER_LR}')
print(f'  Transformer epochs:         {TRANSFORMER_EPOCHS}')
print(f'  Transformer batch size:     {TRANSFORMER_BATCH_SIZE}')
print(f'  Transformer max samples:    {TRANSFORMER_MAX_TRAIN_SAMPLES}')
print(f'  Models enabled:             {[k for k, v in ENABLE_MODELS.items() if v]}')


## 3.0 Data Loading and Preprocessing

This section downloads the transaction dataset from HuggingFace, applies the CRA T2125 category mapping, encodes labels, and creates train/test splits.

The dataset includes transactions from multiple countries and currencies. The models train on all available records regardless of country of origin. Transaction descriptions are language-agnostic text features, so a model trained on a global dataset generalizes better to the varied descriptions that Canadian contractors produce when importing statements from international banks or payment processors.

### 3.1 Load Dataset from HuggingFace

Downloads the Mitul Shah transaction-categorization dataset. The loader handles all available split configurations: pre-split train/test, train/validation, or train-only (auto-split).

All records from all countries are retained. No country filter is applied. If the dataset contains a country or currency column, Section 3.2 reports the distribution for transparency but does not use it to filter rows.

In [ ]:
from datasets import load_dataset

print('[Loading] Downloading dataset from HuggingFace...')
print(f'  Dataset: {DATASET_NAME}')

try:
    load_kwargs = {}
    if HF_TOKEN:
        load_kwargs['token'] = HF_TOKEN

    dataset = load_dataset(DATASET_NAME, **load_kwargs)
    available_splits = list(dataset.keys())
    print(f'[OK] Dataset downloaded. Splits available: {available_splits}')

    if 'test' in available_splits:
        train_data = dataset['train'].to_pandas()
        test_data = dataset['test'].to_pandas()
        print('[OK] Using pre-existing train/test split.')
    elif 'validation' in available_splits:
        train_data = dataset['train'].to_pandas()
        test_data = dataset['validation'].to_pandas()
        print('[OK] Using validation split as test set.')
    else:
        full_data = dataset['train'].to_pandas()
        stratify_col = full_data.columns[-1]
        train_data, test_data = train_test_split(
            full_data,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=full_data[stratify_col],
        )
        print(f'[OK] Created {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} train/test split.')

    print(f'\n  Training samples: {len(train_data):,}')
    print(f'  Test samples:     {len(test_data):,}')
    print(f'  Columns:          {list(train_data.columns)}')
    print(f'  Sample row:\n{train_data.head(2).to_string()}')

    # Report country/currency distribution for transparency -- no filtering applied
    lower_cols = {c.lower(): c for c in train_data.columns}
    for geo_col_candidate in ['country', 'currency', 'region', 'locale']:
        if geo_col_candidate in lower_cols:
            geo_col = lower_cols[geo_col_candidate]
            dist = train_data[geo_col].value_counts()
            print(f'\n  {geo_col} distribution (training set, all retained):')
            for val, count in dist.items():
                pct = count / len(train_data) * 100
                print(f'    {str(val):<20} {count:>6,}  ({pct:.1f}%)')
            break

except Exception as e:
    print(f'[ERROR] Failed to load dataset: {e}')
    raise


### 3.2 Identify Text and Label Columns

Automatically detects the description/text column and the category/label column from the dataset. If a country or currency column exists, it is reported in Section 3.1 for transparency. The models treat transaction descriptions as plain text and do not receive country or currency as a feature, keeping the classifier general-purpose across all geographies.

In [ ]:
try:
    # Identify columns by common naming conventions
    lower_cols = {c.lower(): c for c in train_data.columns}

    # Text column candidates
    text_candidates = ['description', 'text', 'narration', 'transaction', 'details', 'memo']
    text_col = next(
        (lower_cols[c] for c in text_candidates if c in lower_cols),
        train_data.columns[0]
    )

    # Label column candidates
    label_candidates = ['category', 'label', 'class', 'type', 'tag']
    label_col = next(
        (lower_cols[c] for c in label_candidates if c in lower_cols),
        train_data.columns[-1]
    )

    print(f"[OK] Text column:  '{text_col}'")
    print(f"[OK] Label column: '{label_col}'")

    # Confirm no country filtering -- report which columns are excluded from features
    geo_candidates = ['country', 'currency', 'region', 'locale', 'amount']
    excluded = [lower_cols[c] for c in geo_candidates if c in lower_cols]
    if excluded:
        print(f'[INFO] Columns present but excluded from model features: {excluded}')
        print('       Models train on transaction text only for maximum generalizability.')

    print(f'\n  Unique categories ({train_data[label_col].nunique()}):')
    for cat, count in train_data[label_col].value_counts().items():
        print(f'    {cat:<40} {count:>6,}')

except Exception as e:
    print(f'[ERROR] Column detection failed: {e}')
    raise


### 3.3 CRA T2125 Category Mapping

Maps dataset transaction categories to their corresponding CRA T2125 (Statement of Business or Professional Activities) line numbers and deductibility rules.

T2125 is the relevant form for independent contractors and small business owners reporting business income and expenses to the CRA. Each category maps to a specific T2125 line, a deductibility rule, and documentation notes.

Each entry includes an `expense_type` field:
- `current`: Deductible in full in the tax year incurred (e.g., office supplies, software subscriptions, meals).
- `capital`: Must be claimed through Capital Cost Allowance (CCA) over multiple years (e.g., computers, equipment, vehicles). The CCA class determines the annual depreciation rate.

In [ ]:
CRA_MAPPING = {
    "Food & Dining": {
        "cra_line": 8523,
        "cra_form": "T2125 Part 5",
        "cra_description": "Meals and Entertainment",
        "deductibility": "50% deductible (business meals and client entertainment only)",
        "expense_type": "current",
        "notes": "Must be for business purposes; retain itemized receipts. Line 8523 is the current Part 5 standard.",
    },
    "Transportation": {
        "cra_line": 9270,
        "cra_form": "T2125",
        "cra_description": "Motor Vehicle Expenses",
        "deductibility": "Deductible proportional to business use percentage",
        "expense_type": "current",
        "notes": "Covers fuel, insurance, maintenance, and registration. Maintain a mileage logbook. A vehicle purchase is a capital expense claimed via CCA Class 10 (30%) or 10.1 (30%) depending on cost.",
    },
    "Shopping & Retail": {
        "cra_line": 8811,
        "cra_form": "T2125",
        "cra_description": "Office Supplies",
        "deductibility": "Fully deductible for business-use consumable items",
        "expense_type": "current",
        "notes": "Line 8811 covers consumable supplies (paper, pens, printer ink). Durable goods over $500 may qualify as capital expenses under CCA Class 8 (20%).",
    },
    "Entertainment & Recreation": {
        "cra_line": 8523,
        "cra_form": "T2125 Part 5",
        "cra_description": "Meals and Entertainment",
        "deductibility": "50% deductible (business events and client entertainment only)",
        "expense_type": "current",
        "notes": "Client entertainment must have a clear business purpose. Detailed records of attendees and purpose are required.",
    },
    "Healthcare & Medical": {
        "cra_line": None,
        "cra_form": "T2125 (limited)",
        "cra_description": "Not directly deductible on T2125",
        "deductibility": "Generally not deductible as a business expense for sole proprietors",
        "expense_type": "current",
        "notes": "Personal medical expenses are claimed as a non-refundable credit on the T1. Group health insurance premiums for employees can be deducted on T2125 Line 8690.",
    },
    "Utilities & Services": {
        "cra_line": 8810,
        "cra_form": "T2125",
        "cra_description": "Office Expenses",
        "deductibility": "Proportional to business use",
        "expense_type": "current",
        "notes": "Line 8810 covers recurring office-related expenses. Home office utilities are deductible at the business-use percentage of your home.",
    },
    "Financial Services": {
        "cra_line": 8710,
        "cra_form": "T2125",
        "cra_description": "Interest and Bank Charges",
        "deductibility": "Fully deductible for business accounts",
        "expense_type": "current",
        "notes": "Includes bank fees, wire transfer fees, and business credit card charges. Personal account fees are not deductible.",
    },
    "Income": {
        "cra_line": None,
        "cra_form": "T2125 / T1",
        "cra_description": "Business or Employment Income",
        "deductibility": "Not an expense",
        "expense_type": "n/a",
        "notes": "Report as gross business income on T2125 Part 1, or as T4 employment income on the T1.",
    },
    "Travel": {
        "cra_line": 9200,
        "cra_form": "T2125",
        "cra_description": "Travel Expenses",
        "deductibility": "Fully deductible for business travel",
        "expense_type": "current",
        "notes": "Covers airfare, accommodation, and meals during business travel. Personal travel is not deductible. Meals during travel are subject to the 50% limit.",
    },
    "Technology & Software": {
        "cra_line": 8810,
        "cra_form": "T2125",
        "cra_description": "Office Expenses (subscriptions) / CCA Class 12 or 8 (hardware)",
        "deductibility": "Subscriptions: fully deductible current expense. Hardware: capital expense via CCA.",
        "expense_type": "mixed",
        "notes": "SaaS and software subscriptions are current expenses on Line 8810. Computers and hardware are capital expenses: CCA Class 12 (100%, if cost <= $1,800) or Class 8 (20% declining balance for higher-cost equipment). A new MacBook is a capital expense.",
    },
    "Professional Services": {
        "cra_line": 8860,
        "cra_form": "T2125",
        "cra_description": "Legal, Accounting, and Professional Fees",
        "deductibility": "Fully deductible",
        "expense_type": "current",
        "notes": "Accounting, legal, and consulting fees for business purposes. Fees for personal legal matters are not deductible.",
    },
    "Rent & Lease": {
        "cra_line": 8910,
        "cra_form": "T2125",
        "cra_description": "Rent",
        "deductibility": "Fully deductible for business premises",
        "expense_type": "current",
        "notes": "Office or workspace rent is a current expense. Home office rent is proportional to the business-use area. Equipment leases may be current or capital depending on the lease terms.",
    },
    "Insurance": {
        "cra_line": 8690,
        "cra_form": "T2125",
        "cra_description": "Insurance",
        "deductibility": "Fully deductible for business insurance",
        "expense_type": "current",
        "notes": "Business liability, errors and omissions, and commercial property insurance are deductible. Personal life or disability insurance is generally not deductible.",
    },
    "Other": {
        "cra_line": 9270,
        "cra_form": "T2125",
        "cra_description": "Other Expenses",
        "deductibility": "Case-by-case",
        "expense_type": "current",
        "notes": "Miscellaneous business expenses with supporting documentation. Capital items within this catch-all must be separated and claimed via CCA.",
    },
}

print(f"[OK] CRA T2125 mapping defined for {len(CRA_MAPPING)} categories.")
print()
print(f"  {'Category':<30} {'Line':<8} {'Type':<10} Description")
print(f"  {'-'*30} {'-'*8} {'-'*10} {'-'*35}")
for cat, v in CRA_MAPPING.items():
    line = str(v["cra_line"]) if v["cra_line"] else "n/a"
    print(f"  {cat:<30} {line:<8} {v['expense_type']:<10} {v['cra_description']}")


### 3.4 Label Encoding

Encodes string category labels to integer indices for model training. Stores the `LabelEncoder` for later inverse transformation and reporting.

In [ ]:
try:
    X_train_text = train_data[text_col].astype(str).values
    X_test_text = test_data[text_col].astype(str).values
    y_train_raw = train_data[label_col].values
    y_test_raw = test_data[label_col].values

    if y_train_raw.dtype == object or str(y_train_raw.dtype).startswith("str"):
        label_encoder = LabelEncoder()
        y_train = label_encoder.fit_transform(y_train_raw)
        y_test = label_encoder.transform(y_test_raw)
        n_classes = len(label_encoder.classes_)
        print(f"[OK] Label encoder fit on {n_classes} classes.")
        print("\nClass index mapping:")
        for idx, cls in enumerate(label_encoder.classes_):
            cra = CRA_MAPPING.get(cls, {}).get("cra_description", "No CRA mapping")
            print(f"  {idx:>2}: {cls:<40} -> {cra}")
    else:
        y_train = y_train_raw.astype(int)
        y_test = y_test_raw.astype(int)
        n_classes = len(np.unique(y_train))
        label_encoder = None
        print(f"[OK] Labels already numeric. {n_classes} classes.")

    print(f"\n[OK] Training set: {len(X_train_text):,} samples")
    print(f"[OK] Test set:     {len(X_test_text):,} samples")
    print(f"[OK] Classes:      {n_classes}")

except Exception as e:
    print(f"[ERROR] Label encoding failed: {e}")
    raise


## 4.0 Classical ML Models

This section trains three classical ML models: Logistic Regression, Linear SVM, and LightGBM. All three use a shared TF-IDF vectorization pipeline defined in Section 4.1.

### 4.1 Shared Helper Functions

Defines reusable helper functions used by all three classical models:
- `build_tfidf_vectorizer`: creates and fits a TF-IDF vectorizer
- `compute_metrics`: computes accuracy, precision, recall, F1, and inference time
- `measure_model_size_mb`: reports saved model size in MB

In [ ]:
def build_tfidf_vectorizer(X_train, max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE):
    """Fit a TF-IDF vectorizer on training text and return (vectorizer, X_train_matrix, X_test_func)."""
    vec = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    X_train_tfidf = vec.fit_transform(X_train)
    print(f"[OK] TF-IDF vectorizer fit. Vocab size: {len(vec.get_feature_names_out()):,}")
    return vec


def compute_metrics(model_name, y_true, y_pred, train_time, inference_time=None):
    """Compute and return a metrics dict for a model's predictions."""
    metrics = {
        "model": model_name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "training_time_s": float(train_time),
        "inference_time_ms": float(inference_time * 1000) if inference_time is not None else None,
    }
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    print(f"  Train time: {metrics['training_time_s']:.2f}s")
    if metrics["inference_time_ms"] is not None:
        print(f"  Inference:  {metrics['inference_time_ms']:.1f}ms (test set)")
    return metrics


def measure_model_size_mb(filepath):
    """Return file size in MB for a saved model, or None if file does not exist."""
    try:
        return round(os.path.getsize(filepath) / 1e6, 2)
    except FileNotFoundError:
        return None


def time_inference(model, X, repetitions=1):
    """Measure inference wall-clock time across the full set. Returns seconds."""
    t0 = time.perf_counter()
    for _ in range(repetitions):
        model.predict(X)
    return (time.perf_counter() - t0) / repetitions


# Shared results store - populated by each model section
all_results = {}
all_predictions = {}

print("[OK] Helper functions defined.")


### 4.2 TF-IDF Vectorization

Fits the shared TF-IDF vectorizer on the training corpus. The resulting sparse matrices are reused by Logistic Regression, Linear SVM, and LightGBM.

In [ ]:
try:
    tfidf = build_tfidf_vectorizer(X_train_text)
    X_train_tfidf = tfidf.transform(X_train_text)
    X_test_tfidf = tfidf.transform(X_test_text)

    print(f"[OK] Train TF-IDF matrix: {X_train_tfidf.shape}")
    print(f"[OK] Test TF-IDF matrix:  {X_test_tfidf.shape}")

except Exception as e:
    print(f"[ERROR] TF-IDF vectorization failed: {e}")
    raise


### 4.3 Logistic Regression

Trains a multinomial Logistic Regression classifier on TF-IDF features.

In [ ]:
if ENABLE_MODELS["logistic_regression"]:
    print("=" * 60)
    print("MODEL: Logistic Regression")
    print("=" * 60)
    try:
        t0 = time.perf_counter()
        lr_model = LogisticRegression(
            max_iter=LR_MAX_ITER, random_state=RANDOM_SEED, n_jobs=-1
        )
        lr_model.fit(X_train_tfidf, y_train)
        train_time = time.perf_counter() - t0

        infer_time = time_inference(lr_model, X_test_tfidf)
        y_pred_lr = lr_model.predict(X_test_tfidf)

        lr_metrics = compute_metrics(
            "Logistic Regression", y_test, y_pred_lr, train_time, infer_time
        )

        # Save model
        lr_path = os.path.join(project_dir, "models", "logistic_regression.pkl")
        with open(lr_path, "wb") as f:
            pickle.dump({"model": lr_model, "vectorizer": tfidf}, f)
        lr_metrics["model_size_mb"] = measure_model_size_mb(lr_path)
        print(f"  Model size: {lr_metrics['model_size_mb']} MB")

        all_results["Logistic Regression"] = lr_metrics
        all_predictions["Logistic Regression"] = y_pred_lr
        print("[OK] Logistic Regression complete.")

    except Exception as e:
        print(f"[ERROR] Logistic Regression failed: {e}")
        all_results["Logistic Regression"] = {
            "model": "Logistic Regression", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["Logistic Regression"] = np.zeros(len(y_test), dtype=int)
else:
    print("[SKIP] Logistic Regression disabled in ENABLE_MODELS.")


### 4.4 Linear SVM

Trains a Linear Support Vector Machine using the same TF-IDF features as Logistic Regression. LinearSVC is significantly faster than kernel SVMs on large sparse text matrices.

In [ ]:
if ENABLE_MODELS["linear_svm"]:
    print("=" * 60)
    print("MODEL: Linear SVM")
    print("=" * 60)
    try:
        t0 = time.perf_counter()
        svm_model = LinearSVC(
            max_iter=SVM_MAX_ITER, random_state=RANDOM_SEED, dual="auto"
        )
        svm_model.fit(X_train_tfidf, y_train)
        train_time = time.perf_counter() - t0

        infer_time = time_inference(svm_model, X_test_tfidf)
        y_pred_svm = svm_model.predict(X_test_tfidf)

        svm_metrics = compute_metrics(
            "Linear SVM", y_test, y_pred_svm, train_time, infer_time
        )

        svm_path = os.path.join(project_dir, "models", "linear_svm.pkl")
        with open(svm_path, "wb") as f:
            pickle.dump({"model": svm_model, "vectorizer": tfidf}, f)
        svm_metrics["model_size_mb"] = measure_model_size_mb(svm_path)
        print(f"  Model size: {svm_metrics['model_size_mb']} MB")

        all_results["Linear SVM"] = svm_metrics
        all_predictions["Linear SVM"] = y_pred_svm
        print("[OK] Linear SVM complete.")

    except Exception as e:
        print(f"[ERROR] Linear SVM failed: {e}")
        all_results["Linear SVM"] = {
            "model": "Linear SVM", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["Linear SVM"] = np.zeros(len(y_test), dtype=int)
else:
    print("[SKIP] Linear SVM disabled in ENABLE_MODELS.")


### 4.5 LightGBM

Trains a LightGBM gradient boosting classifier on dense TF-IDF features. LightGBM is faster than XGBoost on high-dimensional sparse data when converted to dense format.

In [ ]:
if ENABLE_MODELS["lightgbm"]:
    print("=" * 60)
    print("MODEL: LightGBM")
    print("=" * 60)
    try:
        print("[Converting] Sparse TF-IDF to dense array...")
        X_train_lgb = X_train_tfidf.toarray()
        X_test_lgb = X_test_tfidf.toarray()
        print(f"[OK] Dense arrays: train={X_train_lgb.shape}, test={X_test_lgb.shape}")

        t0 = time.perf_counter()
        lgb_model = LGBMClassifier(
            n_estimators=LGBM_N_ESTIMATORS,
            num_leaves=LGBM_NUM_LEAVES,
            learning_rate=LGBM_LR,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbose=-1,
        )
        lgb_model.fit(X_train_lgb, y_train)
        train_time = time.perf_counter() - t0

        t_inf = time.perf_counter()
        y_pred_lgb = lgb_model.predict(X_test_lgb)
        infer_time = time.perf_counter() - t_inf

        lgb_metrics = compute_metrics(
            "LightGBM", y_test, y_pred_lgb, train_time, infer_time
        )

        lgb_path = os.path.join(project_dir, "models", "lightgbm.pkl")
        with open(lgb_path, "wb") as f:
            pickle.dump({"model": lgb_model, "vectorizer": tfidf}, f)
        lgb_metrics["model_size_mb"] = measure_model_size_mb(lgb_path)
        print(f"  Model size: {lgb_metrics['model_size_mb']} MB")

        all_results["LightGBM"] = lgb_metrics
        all_predictions["LightGBM"] = y_pred_lgb
        print("[OK] LightGBM complete.")

    except MemoryError:
        print("[ERROR] Out of memory converting sparse matrix to dense.")
        print("  Reduce TFIDF_MAX_FEATURES in Section 2.1 and rerun.")
        all_results["LightGBM"] = {
            "model": "LightGBM", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["LightGBM"] = np.zeros(len(y_test), dtype=int)
    except Exception as e:
        print(f"[ERROR] LightGBM failed: {e}")
        all_results["LightGBM"] = {
            "model": "LightGBM", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["LightGBM"] = np.zeros(len(y_test), dtype=int)
else:
    print("[SKIP] LightGBM disabled in ENABLE_MODELS.")


## 5.0 Transformer Models

This section trains two transformer-based models: DistilBERT and MobileBERT. Both use a shared training loop defined in Section 5.1.

### 5.1 Shared Transformer Helper Functions

Defines reusable utilities used by both DistilBERT and MobileBERT:

- `stratified_sample`: Takes a stratified random sample of training data to cap memory use.
- `TransactionDataset`: Lazy PyTorch Dataset that tokenizes one sample at a time in `__getitem__`.
- `train_transformer`: Trains any HuggingFace sequence classification model. Detects
  `HIP error: invalid device function` on the first failing batch and immediately falls back
  to CPU rather than silently skipping every batch and producing zero results.
- `evaluate_transformer`: Batched inference with the same GPU/CPU fallback.
- `save_transformer_model`: Saves model and tokenizer to disk and reports size.

**Note on UNEXPECTED / MISSING key warnings:** When loading `distilbert-base-uncased` or
`google/mobilebert-uncased` for sequence classification, HuggingFace reports UNEXPECTED
keys (the pre-trained MLM head being discarded) and MISSING keys (the new classification
head being randomly initialized). This is correct and expected fine-tuning behavior.

In [ ]:
def stratified_sample(X, y, max_samples, random_seed=RANDOM_SEED):
    """
    Return a stratified random subset of (X, y) capped at max_samples.
    If len(X) <= max_samples the full arrays are returned unchanged.
    """
    if max_samples is None or len(X) <= max_samples:
        return X, y
    _, X_s, _, y_s = train_test_split(
        X, y, test_size=max_samples, stratify=y, random_state=random_seed
    )
    print(f"  [Sample] Using {len(X_s):,} of {len(X):,} training samples (stratified).")
    return X_s, y_s


class TransactionDataset(Dataset):
    """Lazy Dataset: tokenizes one sample at a time in __getitem__, not upfront."""

    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def _is_hip_kernel_error(err):
    """Return True if a RuntimeError is a HIP invalid device function error."""
    msg = str(err).lower()
    return "invalid device function" in msg or ("hip error" in msg and "invalid" in msg)


def train_transformer(
    model,
    tokenizer,
    X_train,
    y_train_arr,
    epochs=TRANSFORMER_EPOCHS,
    batch_size=TRANSFORMER_BATCH_SIZE,
    lr=TRANSFORMER_LR,
    max_length=TRANSFORMER_MAX_LENGTH,
    max_train_samples=TRANSFORMER_MAX_TRAIN_SAMPLES,
    device=device,
):
    """
    Train a HuggingFace sequence classification model.
    On HIP kernel errors (MI300X arch mismatch) falls back to CPU automatically.
    """
    X_tr, y_tr = stratified_sample(X_train, y_train_arr, max_train_samples)
    print(f"  [Dataset] {len(X_tr):,} samples  |  {epochs} epochs  |  batch {batch_size}  |  device {device}")

    train_ds = TransactionDataset(X_tr, y_tr, tokenizer, max_length)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps,
    )

    try:
        model.gradient_checkpointing_enable()
    except Exception:
        pass

    model.to(device)
    model.train()

    active_device = device
    t0 = time.perf_counter()

    for epoch in range(epochs):
        epoch_loss = 0.0
        batches_ok = 0
        for batch in tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs}", leave=False):
            optimizer.zero_grad()
            try:
                input_ids = batch["input_ids"].to(active_device)
                attention_mask = batch["attention_mask"].to(active_device)
                labels_batch = batch["labels"].to(active_device)
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels_batch)
                loss = outputs.loss
                epoch_loss += loss.item()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                batches_ok += 1
            except RuntimeError as batch_err:
                if _is_hip_kernel_error(batch_err) and active_device.type == "cuda":
                    print(f"\n  [WARNING] HIP kernel error on GPU (gfx942 arch mismatch).")
                    print("  [INFO] Falling back to CPU for the remainder of training...")
                    active_device = torch.device("cpu")
                    model.to(active_device)
                    # Retry this batch on CPU
                    try:
                        input_ids = batch["input_ids"].to(active_device)
                        attention_mask = batch["attention_mask"].to(active_device)
                        labels_batch = batch["labels"].to(active_device)
                        outputs = model(input_ids, attention_mask=attention_mask, labels=labels_batch)
                        loss = outputs.loss
                        epoch_loss += loss.item()
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                        scheduler.step()
                        batches_ok += 1
                    except Exception as cpu_err:
                        print(f"  [ERROR] CPU retry also failed: {cpu_err}")
                else:
                    print(f"\n  [WARNING] Batch skipped: {batch_err}")

        avg_loss = epoch_loss / max(batches_ok, 1)
        print(f"  Epoch {epoch+1}/{epochs} -- avg loss: {avg_loss:.4f}  ({batches_ok} batches completed on {active_device})")

    train_time = time.perf_counter() - t0
    return model, train_time, active_device


def evaluate_transformer(
    model,
    tokenizer,
    X_test,
    y_test_arr,
    batch_size=TRANSFORMER_BATCH_SIZE,
    max_length=TRANSFORMER_MAX_LENGTH,
    device=device,
):
    """Evaluate a trained model. Returns (predictions, inference_time_s)."""
    print(f"  [Eval] {len(X_test):,} test samples on {device}...")
    test_ds = TransactionDataset(X_test, list(y_test_arr), tokenizer, max_length)
    test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=0)

    model.to(device)
    model.eval()
    preds = []
    active_device = device
    t0 = time.perf_counter()

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="  Evaluating", leave=False):
            try:
                input_ids = batch["input_ids"].to(active_device)
                attention_mask = batch["attention_mask"].to(active_device)
                outputs = model(input_ids, attention_mask=attention_mask)
                batch_preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
                preds.extend(batch_preds)
            except RuntimeError as batch_err:
                if _is_hip_kernel_error(batch_err) and active_device.type == "cuda":
                    print(f"\n  [INFO] HIP error during eval -- retrying batch on CPU.")
                    active_device = torch.device("cpu")
                    model.to(active_device)
                    try:
                        input_ids = batch["input_ids"].to(active_device)
                        attention_mask = batch["attention_mask"].to(active_device)
                        outputs = model(input_ids, attention_mask=attention_mask)
                        batch_preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
                        preds.extend(batch_preds)
                    except Exception as cpu_err:
                        print(f"  [ERROR] CPU eval retry failed: {cpu_err}")
                else:
                    print(f"\n  [WARNING] Eval batch skipped: {batch_err}")

    infer_time = time.perf_counter() - t0
    return np.array(preds), infer_time


def save_transformer_model(model, tokenizer, save_dir):
    """Save model and tokenizer to disk. Returns size in MB."""
    try:
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        total = sum(
            os.path.getsize(os.path.join(r, f))
            for r, _, files in os.walk(save_dir)
            for f in files
        )
        size_mb = round(total / 1e6, 1)
        print(f"  [OK] Saved to {save_dir} ({size_mb} MB)")
        return size_mb
    except Exception as e:
        print(f"  [WARNING] Could not save model: {e}")
        return None


print("[OK] Transformer helper functions defined.")
print(f"  Max transformer training samples: {TRANSFORMER_MAX_TRAIN_SAMPLES:,}" if TRANSFORMER_MAX_TRAIN_SAMPLES else "  Max transformer training samples: unlimited")


### 5.2 DistilBERT

Fine-tunes `distilbert-base-uncased` for sequence classification. DistilBERT retains 97% of BERT's performance at 40% fewer parameters (~66M parameters, ~265 MB).

In [ ]:
if ENABLE_MODELS["distilbert"]:
    print("=" * 60)
    print("MODEL: DistilBERT")
    print("=" * 60)
    print(f"  Device: {device}  |  Epochs: {TRANSFORMER_EPOCHS}  |  Batch: {TRANSFORMER_BATCH_SIZE}")

    try:
        print("\n[Loading] distilbert-base-uncased from HuggingFace...")
        distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        distilbert_model = DistilBertForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", num_labels=n_classes
        )
        print("[OK] DistilBERT loaded.")

        distilbert_model, train_time, active_device = train_transformer(
            distilbert_model, distilbert_tokenizer,
            X_train_text, y_train
        )
        print(f"[OK] Training complete in {train_time:.1f}s")

        y_pred_distilbert, infer_time = evaluate_transformer(
            distilbert_model, distilbert_tokenizer,
            X_test_text, y_test, device=active_device
        )

        distilbert_metrics = compute_metrics(
            "DistilBERT", y_test, y_pred_distilbert, train_time, infer_time
        )

        # Save model
        distilbert_dir = os.path.join(project_dir, "models", "distilbert")
        distilbert_metrics["model_size_mb"] = save_transformer_model(
            distilbert_model, distilbert_tokenizer, distilbert_dir
        )

        all_results["DistilBERT"] = distilbert_metrics
        all_predictions["DistilBERT"] = y_pred_distilbert
        print("[OK] DistilBERT complete.")

        # Free GPU memory before next model
        del distilbert_model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    except MemoryError:
        print("[ERROR] Out of memory training DistilBERT. Try a smaller batch size or use CPU.")
        all_results["DistilBERT"] = {
            "model": "DistilBERT", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["DistilBERT"] = np.zeros(len(y_test), dtype=int)
    except Exception as e:
        print(f"[ERROR] DistilBERT training failed: {e}")
        all_results["DistilBERT"] = {
            "model": "DistilBERT", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["DistilBERT"] = np.zeros(len(y_test), dtype=int)
else:
    print("[SKIP] DistilBERT disabled in ENABLE_MODELS.")


### 5.3 MobileBERT

Fine-tunes `google/mobilebert-uncased` for sequence classification. MobileBERT is a task-agnostic compressed BERT designed for on-device inference (~24M parameters, ~96 MB). It uses bottleneck structures and inverted-bottleneck feed-forward layers to match BERT's accuracy at significantly lower latency on mobile hardware.

In [ ]:
if ENABLE_MODELS["mobilebert"]:
    print("=" * 60)
    print("MODEL: MobileBERT")
    print("=" * 60)
    print(f"  Device: {device}  |  Epochs: {TRANSFORMER_EPOCHS}  |  Batch: {TRANSFORMER_BATCH_SIZE}")
    print("  Model: google/mobilebert-uncased (~96 MB, optimized for on-device inference)")

    try:
        print("\n[Loading] google/mobilebert-uncased from HuggingFace...")
        mobilebert_tokenizer = MobileBertTokenizer.from_pretrained("google/mobilebert-uncased")
        mobilebert_model = MobileBertForSequenceClassification.from_pretrained(
            "google/mobilebert-uncased",
            num_labels=n_classes,
        )
        param_count = sum(p.numel() for p in mobilebert_model.parameters())
        print(f"[OK] MobileBERT loaded. Parameters: {param_count:,}")

        mobilebert_model, train_time, active_device = train_transformer(
            mobilebert_model, mobilebert_tokenizer,
            X_train_text, y_train
        )
        print(f"[OK] Training complete in {train_time:.1f}s")

        y_pred_mobilebert, infer_time = evaluate_transformer(
            mobilebert_model, mobilebert_tokenizer,
            X_test_text, y_test, device=active_device
        )

        mobilebert_metrics = compute_metrics(
            "MobileBERT", y_test, y_pred_mobilebert, train_time, infer_time
        )

        # Save model
        mobilebert_dir = os.path.join(project_dir, "models", "mobilebert")
        mobilebert_metrics["model_size_mb"] = save_transformer_model(
            mobilebert_model, mobilebert_tokenizer, mobilebert_dir
        )

        all_results["MobileBERT"] = mobilebert_metrics
        all_predictions["MobileBERT"] = y_pred_mobilebert
        print("[OK] MobileBERT complete.")

        # Free GPU memory
        del mobilebert_model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    except MemoryError:
        print("[ERROR] Out of memory training MobileBERT. Try a smaller batch size.")
        all_results["MobileBERT"] = {
            "model": "MobileBERT", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["MobileBERT"] = np.zeros(len(y_test), dtype=int)
    except Exception as e:
        print(f"[ERROR] MobileBERT training failed: {e}")
        all_results["MobileBERT"] = {
            "model": "MobileBERT", "accuracy": 0, "precision": 0,
            "recall": 0, "f1": 0, "training_time_s": 0, "inference_time_ms": None,
            "model_size_mb": None
        }
        all_predictions["MobileBERT"] = np.zeros(len(y_test), dtype=int)
else:
    print("[SKIP] MobileBERT disabled in ENABLE_MODELS.")


## 6.0 Model Comparison

This section compiles results from all five models into a unified comparison table, generates visualizations, and answers the three research questions based on actual results.

### 6.1 Build Comparison DataFrame

Assembles all collected metrics into a `pandas` DataFrame for side-by-side analysis. Adds a browser/mobile suitability column based on model size thresholds.

In [ ]:
try:
    comparison_df = pd.DataFrame(list(all_results.values()))
    comparison_df = comparison_df.set_index("model")

    # Add browser / mobile suitability assessment
    def suitability(row):
        size = row.get("model_size_mb")
        if size is None:
            return "Unknown"
        if size < 5:
            return "Excellent (browser + mobile)"
        elif size < 100:
            return "Good (mobile, limited browser)"
        elif size < 300:
            return "Fair (desktop only; needs quantization for mobile)"
        else:
            return "Poor (too large for on-device)"

    comparison_df["deployment_suitability"] = comparison_df.apply(suitability, axis=1)

    # Numeric columns for display
    display_cols = ["accuracy", "precision", "recall", "f1",
                    "training_time_s", "inference_time_ms", "model_size_mb"]
    print("=" * 80)
    print("MODEL COMPARISON RESULTS")
    print("=" * 80)
    print(comparison_df[display_cols].round(4).to_string())
    print("\nDeployment Suitability:")
    print(comparison_df[["deployment_suitability"]].to_string())

    best_model_name = comparison_df["f1"].idxmax()
    best_f1 = comparison_df.loc[best_model_name, "f1"]
    print(f"\n[BEST MODEL by F1] {best_model_name} (F1={best_f1:.4f})")

    # Save comparison CSV
    csv_path = os.path.join(project_dir, "results", "model_comparison.csv")
    comparison_df.to_csv(csv_path)
    print(f"[OK] Saved to {csv_path}")

    # Save comparison JSON
    json_path = os.path.join(project_dir, "results", "model_comparison.json")
    comparison_df.reset_index().to_json(json_path, orient="records", indent=2)
    print(f"[OK] Saved to {json_path}")

except Exception as e:
    print(f"[ERROR] Comparison DataFrame build failed: {e}")
    raise


### 6.2 Performance Visualization

Four-panel chart comparing accuracy, F1 score, precision/recall, and inference time. A radar chart shows the multi-dimensional performance profile of each model.

In [ ]:
try:
    numeric_df = comparison_df[["accuracy", "precision", "recall", "f1",
                                 "training_time_s", "inference_time_ms"]].copy()
    models = numeric_df.index.tolist()
    colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple", "darkorange"]
    color_map = {m: colors[i % len(colors)] for i, m in enumerate(models)}

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Canadian Transaction Classifier - Model Comparison", fontsize=15, fontweight="bold")

    # 1. Accuracy
    ax = axes[0, 0]
    vals = numeric_df["accuracy"].sort_values()
    bars = ax.barh(vals.index, vals.values, color=[color_map[m] for m in vals.index])
    ax.set_xlabel("Accuracy")
    ax.set_title("Accuracy by Model")
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, vals.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)

    # 2. F1 Score
    ax = axes[0, 1]
    vals = numeric_df["f1"].sort_values()
    bars = ax.barh(vals.index, vals.values, color=[color_map[m] for m in vals.index])
    ax.set_xlabel("Weighted F1 Score")
    ax.set_title("F1 Score by Model")
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, vals.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)

    # 3. Precision vs Recall
    ax = axes[1, 0]
    x = np.arange(len(models))
    width = 0.35
    ax.bar(x - width / 2, numeric_df.loc[models, "precision"], width,
           label="Precision", color="skyblue")
    ax.bar(x + width / 2, numeric_df.loc[models, "recall"], width,
           label="Recall", color="lightcoral")
    ax.set_ylabel("Score")
    ax.set_title("Precision vs Recall")
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha="right")
    ax.legend()
    ax.set_ylim(0, 1.1)

    # 4. Inference Time
    ax = axes[1, 1]
    infer_vals = numeric_df["inference_time_ms"].dropna().sort_values()
    if len(infer_vals) > 0:
        bars = ax.barh(infer_vals.index, infer_vals.values,
                       color=[color_map[m] for m in infer_vals.index])
        ax.set_xlabel("Inference Time (ms, full test set)")
        ax.set_title("Inference Speed by Model")
        for bar, val in zip(bars, infer_vals.values):
            ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{val:.1f}ms", va="center", fontsize=9)
    else:
        ax.text(0.5, 0.5, "No inference time data", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title("Inference Speed by Model")

    plt.tight_layout()
    chart_path = os.path.join(project_dir, "charts", "model_comparison.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    print(f"[OK] Chart saved to {chart_path}")
    plt.show()

except Exception as e:
    print(f"[ERROR] Visualization failed: {e}")


### 6.3 Radar Chart - Multi-Dimensional Profile

A radar (spider) chart shows the overall profile of each model across accuracy, F1, precision, recall, and deployment suitability (inverse of model size).

In [ ]:
try:
    from matplotlib.patches import FancyArrowPatch
    import matplotlib.patches as mpatches

    metrics_for_radar = ["accuracy", "precision", "recall", "f1"]
    # Add normalized deployment score (inverse of size, capped)
    comparison_df["deploy_score"] = comparison_df["model_size_mb"].apply(
        lambda x: max(0, 1 - x / 300) if x is not None and not pd.isna(x) else 0.5
    )
    radar_metrics = metrics_for_radar + ["deploy_score"]
    radar_labels = ["Accuracy", "Precision", "Recall", "F1 Score", "Deploy Score"]

    N = len(radar_metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
    ax.set_title("Model Multi-Dimensional Profile", pad=20, fontsize=13, fontweight="bold")

    colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple", "darkorange"]
    legend_handles = []
    for i, model_name in enumerate(comparison_df.index):
        values = comparison_df.loc[model_name, radar_metrics].tolist()
        values += values[:1]
        color = colors[i % len(colors)]
        ax.plot(angles, values, "o-", linewidth=1.5, color=color, label=model_name)
        ax.fill(angles, values, alpha=0.08, color=color)
        legend_handles.append(mpatches.Patch(color=color, label=model_name))

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=11)
    ax.set_ylim(0, 1)
    ax.yaxis.set_tick_params(labelsize=8)
    ax.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(1.3, 1.15), fontsize=9)

    radar_path = os.path.join(project_dir, "charts", "radar_chart.png")
    plt.savefig(radar_path, dpi=150, bbox_inches="tight")
    print(f"[OK] Radar chart saved to {radar_path}")
    plt.show()

except Exception as e:
    print(f"[WARNING] Radar chart failed: {e}")


### 6.4 Detailed Classification Reports

Prints a per-class precision/recall/F1 report for every model that produced non-trivial predictions.

In [ ]:
class_names = label_encoder.classes_.tolist() if label_encoder is not None else None

for model_name, y_pred in all_predictions.items():
    if all_results.get(model_name, {}).get("f1", 0) > 0:
        print("=" * 60)
        print(f"Classification Report: {model_name}")
        print("=" * 60)
        try:
            report = classification_report(
                y_test, y_pred,
                target_names=class_names,
                zero_division=0
            )
            print(report)
        except Exception as e:
            print(f"[WARNING] Could not generate report for {model_name}: {e}")


### 6.5 Research Question Answers

This cell prints an evidence-based answer to each of the three research questions using the actual comparison results computed above.

In [ ]:
try:
    print("=" * 70)
    print("RESEARCH QUESTION ANSWERS")
    print("=" * 70)

    trained = comparison_df[comparison_df["f1"] > 0]
    if trained.empty:
        print("[WARNING] No models produced valid results. Run all model sections first.")
    else:
        classical = ["Logistic Regression", "Linear SVM", "LightGBM"]
        transformer = ["DistilBERT", "MobileBERT"]

        classical_df = trained.loc[trained.index.intersection(classical)]
        transformer_df = trained.loc[trained.index.intersection(transformer)]

        best_classical = classical_df["f1"].idxmax() if not classical_df.empty else None
        best_transformer = transformer_df["f1"].idxmax() if not transformer_df.empty else None
        best_overall = trained["f1"].idxmax()

        print("\nQ1: Will transformers outperform classical ML?")
        if best_classical and best_transformer:
            cf1 = classical_df.loc[best_classical, "f1"]
            tf1 = transformer_df.loc[best_transformer, "f1"]
            delta = tf1 - cf1
            if delta > 0.01:
                print(f"  FINDING: Yes. Best transformer ({best_transformer}, F1={tf1:.4f}) outperforms")
                print(f"  best classical model ({best_classical}, F1={cf1:.4f}) by {delta:.4f}.")
            elif delta < -0.01:
                print(f"  FINDING: No. Best classical model ({best_classical}, F1={cf1:.4f}) outperforms")
                print(f"  best transformer ({best_transformer}, F1={tf1:.4f}) by {abs(delta):.4f}.")
                print(f"  Classical ML is sufficient and far more efficient for this task.")
            else:
                print(f"  FINDING: Comparable. Classical ({best_classical}, F1={cf1:.4f}) and")
                print(f"  transformer ({best_transformer}, F1={tf1:.4f}) perform similarly.")
                print(f"  Classical ML is preferred for deployment due to size and speed.")
        else:
            print(f"  FINDING: Insufficient data (not all model groups trained).")

        print("\nQ2: Which model is best for CRA T1 expense classification?")
        print(f"  FINDING: {best_overall} achieves the best weighted F1 score")
        print(f"  ({trained.loc[best_overall, 'f1']:.4f}) on the test set.")
        print(f"  It correctly classifies the most expense categories required for T1/T2125 reporting.")

        print("\nQ3: Which model performs best for browser and on-device use (iPhone, Mac)?")
        # Rank by deploy_score or model_size
        if "deploy_score" in trained.columns:
            best_deploy = trained["deploy_score"].idxmax()
        elif "model_size_mb" in trained.columns:
            size_valid = trained["model_size_mb"].dropna()
            best_deploy = size_valid.idxmin() if not size_valid.empty else best_overall
        else:
            best_deploy = best_overall

        size = trained.loc[best_deploy, "model_size_mb"] if "model_size_mb" in trained.columns else None
        print(f"  FINDING: {best_deploy} is best suited for browser (via onnxruntime-web) and")
        if size:
            print(f"  on-device (iOS/macOS) deployment at {size} MB.")
        print(f"  Classical ML models (<5 MB after ONNX export) are highly suitable for")
        print(f"  browser inference with zero backend. MobileBERT (~96 MB) can run on device")
        print(f"  after INT8 quantization (~25-30 MB). DistilBERT (~265 MB) is best for server.")

        print("\n[RECOMMENDATION]")
        print(f"  Production (accuracy-first): {best_overall}")
        print(f"  Browser / mobile (size-first): {best_deploy}")

except Exception as e:
    print(f"[ERROR] Research question analysis failed: {e}")


## 7.0 ONNX Export

This section exports the best-performing model to ONNX format with INT8 quantization for use in:
- **macOS / iOS** via `onnxruntime` (Swift / Objective-C)
- **Next.js web app** (no backend) via `onnxruntime-web` running in the browser
- **Python** inference pipeline for server-side use

### 7.1 Export Classical ML Model to ONNX

If the best model is a classical ML model (LR, SVM, or LightGBM), this cell uses `skl2onnx` to convert the fitted scikit-learn / LightGBM pipeline to ONNX.

In [ ]:
CLASSICAL_MODEL_MAP = {
    "Logistic Regression": ("lr_model", "tfidf"),
    "Linear SVM": ("svm_model", "tfidf"),
    "LightGBM": ("lgb_model", "tfidf"),
}

try:
    best_model_name = comparison_df["f1"].idxmax()
    print(f"[INFO] Best model: {best_model_name}")
    onnx_exported = False

    if best_model_name in CLASSICAL_MODEL_MAP:
        model_var, vec_var = CLASSICAL_MODEL_MAP[best_model_name]
        clf = locals().get(model_var) or globals().get(model_var)
        vec = locals().get(vec_var) or globals().get(vec_var)

        if clf is None or vec is None:
            raise ValueError(f"Model '{model_var}' or vectorizer '{vec_var}' not found in scope.")

        print(f"[Exporting] {best_model_name} to ONNX via skl2onnx...")
        n_features = X_train_tfidf.shape[1]
        initial_type = [("float_input", FloatTensorType([None, n_features]))]

        onnx_model = convert_sklearn(clf, initial_types=initial_type)
        onnx_path = os.path.join(project_dir, "models", "best_model.onnx")
        with open(onnx_path, "wb") as f:
            f.write(onnx_model.SerializeToString())
        size_mb = round(os.path.getsize(onnx_path) / 1e6, 2)
        print(f"[OK] ONNX model saved: {onnx_path} ({size_mb} MB)")

        # Also save TF-IDF vectorizer
        vec_path = os.path.join(project_dir, "models", "tfidf_vectorizer.pkl")
        with open(vec_path, "wb") as f:
            pickle.dump(vec, f)
        print(f"[OK] TF-IDF vectorizer saved: {vec_path}")
        onnx_exported = True

    else:
        print(f"[INFO] Best model ({best_model_name}) is a transformer.")
        print("[INFO] Use Section 7.2 for transformer ONNX export.")

except Exception as e:
    print(f"[ERROR] Classical ONNX export failed: {e}")


### 7.2 Export Transformer Model to ONNX

If the best model is a transformer (DistilBERT or MobileBERT), this cell uses the `optimum` library's `ORTModelForSequenceClassification` to export the model to ONNX with optional INT8 dynamic quantization.

INT8 quantization reduces model weights from 32-bit floats to 8-bit integers. For MobileBERT this shrinks the ONNX file from approximately 96 MB to approximately 25-30 MB -- roughly a 70% reduction -- with minimal accuracy loss (typically under 1% F1). This size reduction directly improves load times and memory usage for iOS users running Core ML inference on-device.

Since transaction data is financial and sensitive, on-device inference means raw bank descriptions never leave the user's device and never touch your servers. This is a meaningful privacy guarantee for Canadian users under PIPEDA.

In [ ]:
TRANSFORMER_MODEL_MAP = {
    "DistilBERT": os.path.join(project_dir, "models", "distilbert"),
    "MobileBERT": os.path.join(project_dir, "models", "mobilebert"),
}

try:
    best_model_name = comparison_df["f1"].idxmax()

    if best_model_name in TRANSFORMER_MODEL_MAP:
        saved_dir = TRANSFORMER_MODEL_MAP[best_model_name]
        onnx_out_dir = os.path.join(project_dir, "models", "onnx_transformer")

        if not os.path.exists(saved_dir):
            print(f"[WARNING] Saved transformer dir not found: {saved_dir}")
            print("  Run Section 5 to train and save the transformer model first.")
        else:
            print(f"[Exporting] {best_model_name} to ONNX via optimum...")
            try:
                from optimum.onnxruntime import ORTModelForSequenceClassification
                from transformers import AutoTokenizer

                model_ort = ORTModelForSequenceClassification.from_pretrained(
                    saved_dir, export=True
                )
                tokenizer_ort = AutoTokenizer.from_pretrained(saved_dir)
                os.makedirs(onnx_out_dir, exist_ok=True)
                model_ort.save_pretrained(onnx_out_dir)
                tokenizer_ort.save_pretrained(onnx_out_dir)

                total = sum(
                    os.path.getsize(os.path.join(r, f))
                    for r, _, files in os.walk(onnx_out_dir)
                    for f in files
                )
                print(f"[OK] ONNX model saved to {onnx_out_dir} ({total/1e6:.1f} MB)")

                # INT8 quantization
                print("\n[Quantizing] Applying dynamic INT8 quantization...")
                from optimum.onnxruntime.configuration import AutoQuantizationConfig
                from optimum.onnxruntime import ORTQuantizer

                quantizer = ORTQuantizer.from_pretrained(onnx_out_dir)
                qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
                quant_out_dir = os.path.join(project_dir, "models", "onnx_transformer_int8")
                quantizer.quantize(save_dir=quant_out_dir, quantization_config=qconfig)
                total_q = sum(
                    os.path.getsize(os.path.join(r, f))
                    for r, _, files in os.walk(quant_out_dir)
                    for f in files
                )
                print(f"[OK] Quantized ONNX saved to {quant_out_dir} ({total_q/1e6:.1f} MB)")
                print(f"[OK] Size reduction: {(1 - total_q/total)*100:.1f}%")

            except ImportError:
                print("[WARNING] optimum not installed. Trying torch.onnx.export fallback...")
                # Fallback: manual export with torch.onnx
                try:
                    from transformers import AutoTokenizer, AutoModelForSequenceClassification
                    tokenizer_fb = AutoTokenizer.from_pretrained(saved_dir)
                    model_fb = AutoModelForSequenceClassification.from_pretrained(saved_dir)
                    model_fb.eval()
                    dummy_input = tokenizer_fb(
                        ["sample transaction text"],
                        return_tensors="pt", padding=True, truncation=True, max_length=128
                    )
                    onnx_path = os.path.join(project_dir, "models", "transformer_manual.onnx")
                    torch.onnx.export(
                        model_fb,
                        (dummy_input["input_ids"], dummy_input["attention_mask"]),
                        onnx_path,
                        input_names=["input_ids", "attention_mask"],
                        output_names=["logits"],
                        dynamic_axes={
                            "input_ids": {0: "batch", 1: "seq"},
                            "attention_mask": {0: "batch", 1: "seq"},
                            "logits": {0: "batch"},
                        },
                        opset_version=14,
                    )
                    size_mb = round(os.path.getsize(onnx_path) / 1e6, 1)
                    print(f"[OK] Manual ONNX export: {onnx_path} ({size_mb} MB)")
                except Exception as e2:
                    print(f"[ERROR] Fallback ONNX export also failed: {e2}")
    else:
        print(f"[INFO] Best model ({best_model_name}) is not a transformer. Section 7.1 handled export.")

except Exception as e:
    print(f"[ERROR] Transformer ONNX export failed: {e}")


### 7.3 Verify ONNX Model

Runs a quick inference test using `onnxruntime` to confirm the exported ONNX model is valid and produces predictions.

In [ ]:
try:
    onnx_path = os.path.join(project_dir, "models", "best_model.onnx")
    if os.path.exists(onnx_path):
        sess = ort.InferenceSession(onnx_path)
        input_name = sess.get_inputs()[0].name
        print(f"[OK] ONNX session created. Input: '{input_name}'")

        # Classical model needs TF-IDF preprocessing
        sample_texts = ["gas station purchase", "coffee shop", "amazon.ca"]
        vec_path = os.path.join(project_dir, "models", "tfidf_vectorizer.pkl")
        if os.path.exists(vec_path):
            with open(vec_path, "rb") as f:
                tfidf_loaded = pickle.load(f)
            X_sample = tfidf_loaded.transform(sample_texts).toarray().astype(np.float32)
            preds_raw = sess.run(None, {input_name: X_sample})
            pred_labels = preds_raw[0]
            if label_encoder is not None:
                pred_names = label_encoder.inverse_transform(pred_labels)
            else:
                pred_names = pred_labels
            print("\nSample ONNX Predictions:")
            for text, pred in zip(sample_texts, pred_names):
                print(f"  '{text}' -> {pred}")
        else:
            print("[WARNING] TF-IDF vectorizer not found for ONNX test inference.")
    else:
        print("[INFO] No classical ONNX model found at expected path.")
        print(f"  Expected: {onnx_path}")
        onnx_transformer_path = os.path.join(project_dir, "models", "onnx_transformer")
        if os.path.exists(onnx_transformer_path):
            print(f"  Transformer ONNX found at: {onnx_transformer_path}")

except Exception as e:
    print(f"[ERROR] ONNX verification failed: {e}")


### 7.4 Next.js / Browser Integration Notes

Instructions for using the exported ONNX model in a browser-based Next.js application with no backend server.

**Deployment strategy by platform:**
- Browser (Next.js, no backend): Use a classical ML ONNX model (under 5 MB). It loads instantly and runs in WebAssembly via `onnxruntime-web`. No server round-trip required.
- iOS and macOS: Use MobileBERT INT8 ONNX (approximately 25-30 MB) converted to Core ML format. Inference runs on the Neural Engine. Bank data never leaves the device.
- Privacy note: On-device inference is a strong selling point for a Canadian financial app. Under PIPEDA, processing sensitive financial data entirely on the user's device avoids the data-transfer obligations that apply when sending data to a server.

**Browser Deployment (Next.js, no backend)**

Install `onnxruntime-web`:
```bash
npm install onnxruntime-web
```

Load and run the model in a React component or server component:
```typescript
import * as ort from 'onnxruntime-web';

// Place model file in public/models/best_model.onnx
async function classifyTransaction(description: string): Promise<string> {
  const session = await ort.InferenceSession.create('/models/best_model.onnx');

  // For classical models: TF-IDF preprocessing must be replicated in JS.
  // Recommended: pre-compute feature vectors server-side and store as JSON,
  // OR use a lightweight JS TF-IDF library (e.g. natural, compromise).

  const inputTensor = new ort.Tensor('float32', featureVector, [1, 5000]);
  const results = await session.run({ float_input: inputTensor });
  const labelIndex = results['output_label'].data[0];
  return LABEL_MAP[labelIndex]; // map index back to CRA category name
}
```

**Key considerations:**
- Classical ML ONNX models are <5 MB and load instantly in the browser.
- MobileBERT ONNX (INT8) is ~25-30 MB - viable for progressive loading.
- DistilBERT ONNX is ~130 MB quantized - too large for most browsers without streaming.
- For iOS/macOS use `onnxruntime-objc` or convert to Core ML with `coremltools`.

**Core ML conversion (iOS/macOS):**
```bash
pip install coremltools
python -c "
import coremltools as ct
model = ct.convert('best_model.onnx', source='onnx')
model.save('BestModel.mlpackage')
"
```

## 8.0 Results and Report

This section exports all results to CSV and JSON, and generates a final narrative report answering the three research questions.

### 8.1 Export All Results

Saves the final comparison DataFrame and a structured JSON report to the results directory.

In [ ]:
try:
    # Full comparison CSV (including deployment suitability)
    full_csv = os.path.join(project_dir, "results", "model_comparison_full.csv")
    comparison_df.to_csv(full_csv)
    print(f"[OK] Full comparison CSV: {full_csv}")

    # JSON summary report
    report = {
        "project": "CanadianTransactionClassifier",
        "dataset": DATASET_NAME,
        "n_train": int(len(X_train_text)),
        "n_test": int(len(X_test_text)),
        "n_classes": int(n_classes),
        "class_names": label_encoder.classes_.tolist() if label_encoder else [],
        "models": {},
        "best_model_by_f1": comparison_df["f1"].idxmax(),
        "research_questions": {
            "Q1_transformers_vs_classical": "See comparison_df",
            "Q2_best_for_cra_t1": comparison_df["f1"].idxmax(),
            "Q3_best_for_browser_mobile": (
                comparison_df["deploy_score"].idxmax()
                if "deploy_score" in comparison_df.columns
                else "Unknown"
            ),
        },
    }
    for model_name in comparison_df.index:
        row = comparison_df.loc[model_name].to_dict()
        report["models"][model_name] = {
            k: (None if (isinstance(v, float) and pd.isna(v)) else
                (float(v) if isinstance(v, (float, np.floating)) else
                 (int(v) if isinstance(v, (int, np.integer)) else str(v))))
            for k, v in row.items()
        }

    report_path = os.path.join(project_dir, "results", "final_report.json")
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"[OK] JSON report: {report_path}")

except Exception as e:
    print(f"[ERROR] Results export failed: {e}")


### 8.2 Final Summary

Prints a final human-readable summary of all results, the best model recommendation, and next steps.

In [ ]:
try:
    print("=" * 70)
    print("FINAL SUMMARY - Canadian Transaction Classifier")
    print("=" * 70)

    print(f"\nDataset:       {DATASET_NAME}")
    print(f"Training set:  {len(X_train_text):,} samples")
    print(f"Test set:      {len(X_test_text):,} samples")
    print(f"Classes:       {n_classes}")

    trained_models = comparison_df[comparison_df["f1"] > 0]
    print(f"\nModels trained: {len(trained_models)} / {len(comparison_df)}")

    if not trained_models.empty:
        best = trained_models["f1"].idxmax()
        print("\nPerformance Summary (sorted by F1):")
        sorted_df = trained_models.sort_values("f1", ascending=False)
        for model in sorted_df.index:
            r = sorted_df.loc[model]
            size = f"{r['model_size_mb']:.1f} MB" if r.get("model_size_mb") and not pd.isna(r.get("model_size_mb", float('nan'))) else "N/A"
            print(f"  {model:<25} F1={r['f1']:.4f}  Acc={r['accuracy']:.4f}  "
                  f"Time={r['training_time_s']:.1f}s  Size={size}")

        print(f"\nBest model:    {best}")
        print(f"F1 Score:      {trained_models.loc[best, 'f1']:.4f}")
        print(f"Accuracy:      {trained_models.loc[best, 'accuracy']:.4f}")

    print("\nOutput files:")
    for root, dirs, files in os.walk(project_dir):
        level = root.replace(project_dir, "").count(os.sep)
        indent = "  " * level
        for fname in files:
            fpath = os.path.join(root, fname)
            size_kb = os.path.getsize(fpath) / 1024
            print(f"  {indent}{fname}  ({size_kb:.0f} KB)")

    print("\nNext steps:")
    print("  1. Deploy the best model ONNX to Next.js via onnxruntime-web.")
    print("  2. Convert to Core ML for iOS/macOS using coremltools.")
    print("  3. Set up a lightweight preprocessing API if using transformer ONNX in browser.")
    print("  4. Validate predictions against actual CRA T2125 line items.")

    print("\n" + "=" * 70)
    print("[DONE] Project completed.")
    print("=" * 70)

except Exception as e:
    print(f"[ERROR] Final summary failed: {e}")
